# ⚡ SD WebUI Forge (Lobe Theme + Trợ lý Prompt + 4 Pack Plugins Toàn Diện)

Notebook triển khai **SD WebUI Forge** thế hệ mới kết hợp giao diện **Lobe Theme** & bộ **4 Pack Plugins chuyên nghiệp**:
- 🚀 **Nhân SD WebUI Forge mạnh mẽ**: Tốc độ render nhanh hơn tới 40%, kiến trúc tối ưu VRAM chống tràn bộ nhớ (OOM) vượt trội, tích hợp sẵn ControlNet & LayerDiffuse native.
- 💎 **Giao diện Lobe Theme**: Hiện đại phong cách Midjourney / Apple, Dark mode, Glassmorphism, tối giản và chuyên nghiệp.
- 🧠 **Trợ lý Prompt All-In-One (`sd-webui-prompt-all-in-one`)**:
  - 🇻🇳 **Dịch trực tiếp tiếng Việt -> tiếng Anh** ngay trong WebUI, không cần mở Google Dịch.
  - 📚 **Từ điển từ khóa phân loại sẵn**: Khuôn mặt, Tóc, Áo quần, Tư thế, Góc chụp, Ánh sáng, Chất lượng... chỉ cần click chọn.
  - 🎴 **Tự động nhận diện LoRA**: Hiển thị danh sách LoRA có sẵn để bấm 1-click chèn vào prompt kèm thanh chỉnh weight.
- 🧩 **Tích hợp 4 Pack Plugins toàn diện**:
  - **Pack 1 (Core & Quản lý ảnh)**: Tag Autocomplete, Aspect Ratio Helper, Infinite Image Browsing.
  - **Pack 2 (Tinh chỉnh Mặt/Tay, Phân vùng & Upscale 4K)**: ADetailer (phục hồi mặt/tay), Ultimate SD Upscale (nâng nét 4K), Regional Prompter (phân vùng vẽ).
  - **Pack 3 (Tách nền, Tạo Mask tự động & Đọc Tag từ ảnh)**: RemBG tách nền, WD14 Tagger (đọc prompt từ ảnh), Segment Anything (SAM tạo mask thông minh).
  - **Pack 4 (Quản lý Model & Đồng bộ Civitai)**: Civitai Helper (đồng bộ metadata/preview LoRA), SD Model Downloader.
- 💾 **Đồng bộ Google Drive bền vững**: Lưu giữ cấu hình (`configs_forge`), dùng chung kho Model (`MyDrive/ComfyStudio/models`), không lo mất cài đặt khi tắt máy.
- ⚡ **Download đa luồng `aria2c`**: Tải model tốc độ cao, bypass 403 Civitai, chống tải trùng lặp 100%.

In [ ]:
#@title 1. Kết nối Google Drive & Cấu hình Thư mục Dùng chung
import os
import shutil
from google.colab import drive

print("🚀 Đang kết nối Google Drive...")
drive.mount('/content/drive')

# Thư mục gốc lưu trữ model trên Drive (dùng chung với ComfyStudio)
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
DRIVE_OUTPUTS = "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111"

# Các thư mục model cần thiết cho Automatic1111
FOLDERS = [
    "checkpoints",
    "loras",
    "vae",
    "controlnet",
    "upscale_models",
    "embeddings"
]

for f in FOLDERS:
    os.makedirs(os.path.join(DRIVE_BASE, f), exist_ok=True)
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

print(f"✅ Đã sẵn sàng thư mục Models trên Drive: {DRIVE_BASE}")
print(f"📁 Thư mục xuất ảnh tự động: {DRIVE_OUTPUTS}")


In [ ]:
#@title 2. Cài đặt SD WebUI Forge + Lobe Theme + Trợ lý Prompt + 4 Pack Plugins
import os
import shutil
import subprocess

WEBUI_DIR = "/content/stable-diffusion-webui"
EXT_DIR = "/content/stable-diffusion-webui/extensions"

# 1. Clone SD WebUI Forge (giữ nguyên đường dẫn thư mục đích /content/stable-diffusion-webui)
if not os.path.exists(WEBUI_DIR):
    print("📦 Đang clone SD WebUI Forge...")
    !git clone --depth 1 https://github.com/lllyasviel/stable-diffusion-webui-forge.git /content/stable-diffusion-webui
else:
    print("ℹ️ Thư mục WebUI đã tồn tại, cập nhật code mới nhất...")
    %cd /content/stable-diffusion-webui
    !git pull

%cd /content/stable-diffusion-webui

# 2. Cài đặt aria2 (SD Forge tự quản lý bộ nhớ native qua PyTorch, không cần xformers cũ)
print("⚡ Cài đặt aria2...")
!apt-get update -qq && apt-get install -y -qq aria2

# 3. Cài đặt bộ Extensions, 4 Pack Plugins & Giao diện Lobe Theme
os.makedirs(EXT_DIR, exist_ok=True)

# A. Giao diện Lobe Theme (Hiện đại phong cách Apple / Midjourney)
if not os.path.exists(f"{EXT_DIR}/sd-webui-lobe-theme"):
    print("💎 Đang cài đặt Giao diện Lobe Theme...")
    !git clone --depth 1 https://github.com/canisminor1990/sd-webui-lobe-theme.git {EXT_DIR}/sd-webui-lobe-theme

# B. Trợ lý Prompt All-In-One (Dịch tiếng Việt, Từ điển từ khóa phân loại, Quản lý LoRA)
if not os.path.exists(f"{EXT_DIR}/sd-webui-prompt-all-in-one"):
    print("🧠 Đang cài đặt Trợ lý Prompt All-In-One (Dịch thuật + Từ điển tag)...")
    !git clone --depth 1 https://github.com/Physton/sd-webui-prompt-all-in-one.git {EXT_DIR}/sd-webui-prompt-all-in-one

# --- PACK 1: Core & Quản lý ảnh ---
if not os.path.exists(f"{EXT_DIR}/a1111-sd-webui-tagcomplete"):
    print("🔌 [Pack 1] Đang cài đặt Tag Autocomplete...")
    !git clone --depth 1 https://github.com/DominikDoom/a1111-sd-webui-tagcomplete.git {EXT_DIR}/a1111-sd-webui-tagcomplete

if not os.path.exists(f"{EXT_DIR}/sd-webui-aspect-ratio-helper"):
    print("🔌 [Pack 1] Đang cài đặt Aspect Ratio Helper...")
    !git clone --depth 1 https://github.com/thomasasfk/sd-webui-aspect-ratio-helper.git {EXT_DIR}/sd-webui-aspect-ratio-helper

if not os.path.exists(f"{EXT_DIR}/sd-webui-infinite-image-browsing"):
    print("🔌 [Pack 1] Đang cài đặt Infinite Image Browsing...")
    !git clone --depth 1 https://github.com/zanllp/sd-webui-infinite-image-browsing.git {EXT_DIR}/sd-webui-infinite-image-browsing

# --- PACK 2: Tinh chỉnh Mặt/Tay, Phân vùng & Upscale 4K ---
if not os.path.exists(f"{EXT_DIR}/adetailer"):
    print("🔌 [Pack 2] Đang cài đặt ADetailer (Phục hồi chi tiết mặt & tay)...")
    !git clone --depth 1 https://github.com/Bing-su/adetailer.git {EXT_DIR}/adetailer

if not os.path.exists(f"{EXT_DIR}/ultimate-sd-upscale"):
    print("🔌 [Pack 2] Đang cài đặt Ultimate SD Upscale (Nâng nét 4K)...")
    !git clone --depth 1 https://github.com/Coyote-A/ultimate-sd-upscale.git {EXT_DIR}/ultimate-sd-upscale

if not os.path.exists(f"{EXT_DIR}/sd-webui-regional-prompter"):
    print("🔌 [Pack 2] Đang cài đặt Regional Prompter (Phân vùng vẽ)...")
    !git clone --depth 1 https://github.com/hako-mikan/sd-webui-regional-prompter.git {EXT_DIR}/sd-webui-regional-prompter

# --- PACK 3: Tách nền, Tạo Mask tự động & Đọc Tag từ ảnh ---
if not os.path.exists(f"{EXT_DIR}/stable-diffusion-webui-rembg"):
    print("🔌 [Pack 3] Đang cài đặt RemBG (Tách nền tự động)...")
    !git clone --depth 1 https://github.com/AUTOMATIC1111/stable-diffusion-webui-rembg.git {EXT_DIR}/stable-diffusion-webui-rembg

if not os.path.exists(f"{EXT_DIR}/stable-diffusion-webui-wd14-tagger"):
    print("🔌 [Pack 3] Đang cài đặt WD14 Tagger (Đọc prompt từ ảnh)...")
    !git clone --depth 1 https://github.com/toriato/stable-diffusion-webui-wd14-tagger.git {EXT_DIR}/stable-diffusion-webui-wd14-tagger

if not os.path.exists(f"{EXT_DIR}/sd-webui-segment-anything"):
    print("🔌 [Pack 3] Đang cài đặt Segment Anything (SAM Mask)...")
    !git clone --depth 1 https://github.com/continue-revolution/sd-webui-segment-anything.git {EXT_DIR}/sd-webui-segment-anything

# --- PACK 4: Quản lý Model & Đồng bộ Civitai ---
if not os.path.exists(f"{EXT_DIR}/Stable-Diffusion-Webui-Civitai-Helper"):
    print("🔌 [Pack 4] Đang cài đặt Civitai Helper...")
    !git clone --depth 1 https://github.com/zixaphir/Stable-Diffusion-Webui-Civitai-Helper.git {EXT_DIR}/Stable-Diffusion-Webui-Civitai-Helper

if not os.path.exists(f"{EXT_DIR}/sd-model-downloader"):
    print("🔌 [Pack 4] Đang cài đặt SD Model Downloader...")
    !git clone --depth 1 https://github.com/Iyashinouta/sd-model-downloader.git {EXT_DIR}/sd-model-downloader

# 4. Định nghĩa bộ Style Mẫu (styles.csv) để nạp 1-click trong WebUI
styles_content = """name,prompt,negative_prompt
"🔥 Krea 2 / LUSTIFY Succubus","A woman is captured as a melancholic succubus with horns and wings, posed against a backdrop of ethereal white cherry blossoms and cool blue atmospheric lighting, very realistic, 8k, cinematic photography, <lora:KNP_000003000:1.0>","worst quality, low quality, bad anatomy, deformed, distorted, blurry, cartoon, 3d render"
"📷 Realistic Vision V6 (Paris Candid)","RAW street style photography of a beautiful model in Paris, natural candid smile, stylish beige trench coat, soft golden hour sunlight, 50mm f/1.8, bokeh, hyperdetailed skin, authentic look, 8k","deformed, bad anatomy, disfigured, poorly drawn face, mutation, extra limb, ugly, disgusting, blurred, watermark, bad hands, cartoon, 3d render"
"⚡ SDXL Lightning Master (8K Portrait)","Award-winning RAW cinematic portrait of an elegant woman, authentic skin pores, fine skin texture, soft chiaroscuro studio lighting, 85mm f/1.4 lens, natural gaze, luxury atmosphere, photorealistic, 8k uhd, masterpiece, <lora:add-detail-xl:0.8>","ugly, deformed, noisy, blurry, low contrast, cartoon, anime, 3d render, extra limbs, bad hands, plastic skin"
"💎 Asian Beauty Elegance","RAW close-up portrait of a gorgeous 22yo Asian woman, natural glowing skin texture, subtle makeup, traditional silk details, soft cinematic lighting, 85mm lens, 8k uhd, photorealistic","disfigured, ugly, bad art, deformed, poorly drawn, extra limbs, dull eyes, low quality"
"✨ Detail & Skin Booster","hyperdetailed skin texture, visible pores, individual eyelashes, soft rim lighting, sharp focus, 85mm portrait photography, 8k uhd, professional color grading","plastic skin, airbrushed, wax figure, smooth skin, doll, oversaturated"
""".strip()

# 5. Lưu trữ bền vững cấu hình trên Drive & liên kết styles.csv, config.json, ui-config.json
DRIVE_CONFIG = "/content/drive/MyDrive/ComfyStudio/configs_forge"
os.makedirs(DRIVE_CONFIG, exist_ok=True)

# A. styles.csv
drive_styles = os.path.join(DRIVE_CONFIG, "styles.csv")
local_styles = "/content/stable-diffusion-webui/styles.csv"
if not os.path.exists(drive_styles):
    with open(drive_styles, "w", encoding="utf-8") as f:
        f.write(styles_content)
    print("📋 Đã khởi tạo styles.csv mẫu vào Google Drive!")
else:
    print("📋 Đã tìm thấy styles.csv lưu trên Google Drive.")

if os.path.exists(local_styles) and not os.path.islink(local_styles):
    os.remove(local_styles)
if not os.path.exists(local_styles):
    os.symlink(drive_styles, local_styles)
    print(f"🔗 Đã symlink styles.csv: {drive_styles} -> {local_styles}")

# B. config.json & ui-config.json
for cfg_name in ["config.json", "ui-config.json"]:
    drive_cfg = os.path.join(DRIVE_CONFIG, cfg_name)
    local_cfg = os.path.join("/content/stable-diffusion-webui", cfg_name)
    if not os.path.exists(drive_cfg):
        if os.path.exists(local_cfg) and not os.path.islink(local_cfg):
            shutil.move(local_cfg, drive_cfg)
        else:
            with open(drive_cfg, "w", encoding="utf-8") as f:
                f.write("{}")
    if os.path.exists(local_cfg) and not os.path.islink(local_cfg):
        os.remove(local_cfg)
    if not os.path.exists(local_cfg):
        os.symlink(drive_cfg, local_cfg)
        print(f"🔗 Đã symlink {cfg_name}: {drive_cfg} -> {local_cfg}")

# 6. Tạo Symlink liên kết thư mục Models từ Google Drive vào SD WebUI Forge
DRIVE_BASE = "/content/drive/MyDrive/ComfyStudio/models"
LINK_MAP = {
    f"{DRIVE_BASE}/checkpoints": "/content/stable-diffusion-webui/models/Stable-diffusion",
    f"{DRIVE_BASE}/loras": "/content/stable-diffusion-webui/models/Lora",
    f"{DRIVE_BASE}/vae": "/content/stable-diffusion-webui/models/VAE",
    f"{DRIVE_BASE}/controlnet": "/content/stable-diffusion-webui/models/ControlNet",
    f"{DRIVE_BASE}/embeddings": "/content/stable-diffusion-webui/embeddings",
    f"{DRIVE_BASE}/upscale_models": "/content/stable-diffusion-webui/models/ESRGAN"
}

for src, dst in LINK_MAP.items():
    if os.path.exists(src):
        if os.path.exists(dst) and not os.path.islink(dst):
            for item in os.listdir(dst):
                s_path = os.path.join(dst, item)
                d_path = os.path.join(src, item)
                if not os.path.exists(d_path):
                    shutil.move(s_path, d_path)
            shutil.rmtree(dst)
        if not os.path.exists(dst):
            os.symlink(src, dst)
            print(f"🔗 Symlink thành công: {src} -> {dst}")

print("\n✅ Cài đặt hoàn tất! Giao diện Lobe Theme & 4 Pack Plugins đã sẵn sàng.")

In [ ]:
#@title 📦 3. Cài Đặt Model (Fast / Full)
#@markdown Chọn chế độ cài đặt model:
#@markdown - ⚡ **Fast (Setup Cơ bản)**: Chỉ tải 2 Checkpoint cốt lõi (SDXL Turbo + SD1.5 Realistic) + VAE + Upscaler + 1 LoRA để chạy thử ngay lập tức.
#@markdown - 📦 **Full (Trọn bộ Fullpack)**: Quét và tự động tải tất cả các model còn thiếu trong kho model tuyển chọn.
#@markdown - 🛑 **Chống tải trùng 100%**: File đã có trên Drive (`MyDrive/ComfyStudio/models`) sẽ tự động bỏ qua, tuyệt đối không tải lại.

# --- 🎯 CHỌN CHẾ ĐỘ ---
Che_Do = "⚡ Fast (Setup Cơ bản - Chạy nhanh nhất)" #@param ["⚡ Fast (Setup Cơ bản - Chạy nhanh nhất)", "📦 Full (Tải trọn bộ tất cả pack còn thiếu)", "🔍 Kiểm tra trạng thái trên Google Drive"]

# --- 🔐 MẬT KHẨU MÃ HÓA & API TOKENS ---
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}
Civitai_API_Token = "" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

import os
import sys
import subprocess
from huggingface_hub import hf_hub_download

# Cài đặt aria2 nếu chưa có
!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open('/content/.vault_tokens.json', 'w') as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = '', pwd: str = '') -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists('/content/.vault_tokens.json'):
            cached = json.load(open('/content/.vault_tokens.json'))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ''

Civitai_API_Token = get_vault_token('civitai', Civitai_API_Token, Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token('huggingface', HuggingFace_Token, Mat_Khau_Mo_Khoa)
if Civitai_API_Token or HuggingFace_Token:
    print('🔓 [Bảo Mật] Đã mở khóa thành công API Tokens qua Mật khẩu!')

DRIVE_DIR = '/content/drive/MyDrive/ComfyStudio/models'
A1111_DIR = '/content/stable-diffusion-webui/models'
has_drive = os.path.exists('/content/drive/MyDrive')
TARGET_BASE = DRIVE_DIR if has_drive else A1111_DIR

WEBUI_SUBDIR_MAP = {
    'checkpoints': 'Stable-diffusion',
    'loras': 'Lora',
    'vae': 'VAE',
    'controlnet': 'ControlNet',
    'upscale_models': 'ESRGAN',
    'embeddings': '../embeddings'
}

# DANH MỤC MODEL CHUẨN TUYỂN CHỌN
CURATED_MODELS = [
    # --- CHECKPOINTS ---
    {
        'id': 'krea2_turbo',
        'group': 'CHECKPOINTS',
        'is_fast': True,
        'name': 'Krea 2 Turbo / LUSTIFY! NSFW [SDXL 4-Step]',
        'category': 'checkpoints',
        'filename': 'lustify_workflows_krea_2_sdxl.safetensors',
        'aliases': ['lustify', 'krea2', 'krea_2', 'lustifynsfwcheckpoint'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'model_version_id': '3112728',
        'url': 'https://civitai.com/api/download/models/3112728',
        'desc': 'SDXL Turbo cực nét gen 4 bước (~2-4s)'
    },
    {
        'id': 'realistic_vision_v6',
        'group': 'CHECKPOINTS',
        'is_fast': True,
        'name': 'Realistic Vision V6.0 B1 [SD1.5]',
        'category': 'checkpoints',
        'filename': 'realisticVisionV60B1_v51VAE.safetensors',
        'aliases': ['realisticvision', 'realistic_vision', 'v51hypervae', 'v51vae'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'model_version_id': '130072',
        'url': 'https://civitai.com/api/download/models/130072?type=Model&format=SafeTensor&size=pruned&fp=fp16',
        'alt_hf': {'repo_id': 'SG161222/Realistic_Vision_V5.1_noVAE', 'filename': 'Realistic_Vision_V5.1_fp16-no-ema.safetensors'},
        'desc': 'SD1.5 chụp ảnh người thật photorealistic kinh điển'
    },
    {
        'id': 'wai_illustrious',
        'group': 'CHECKPOINTS',
        'is_fast': False,
        'name': 'WAI-illustrious-SDXL [SDXL Anime/Illustrious]',
        'category': 'checkpoints',
        'filename': 'waiIllustriousSDXL_v170.safetensors',
        'aliases': ['waiillustrious', 'wai-illustrious', 'illustrious_sdxl'],
        'min_size_mb': 2000,
        'source_type': 'civitai',
        'model_version_id': '2883731',
        'url': 'https://civitai.com/api/download/models/2883731',
        'desc': 'SDXL minh họa anime & manga đỉnh cao'
    },
    {
        'id': 'majicmix_realistic',
        'group': 'CHECKPOINTS',
        'is_fast': False,
        'name': 'majicMIX realistic 麦橘写实 [SD1.5]',
        'category': 'checkpoints',
        'filename': 'majicmixRealistic_v7.safetensors',
        'aliases': ['majicmix', 'majicmixrealistic'],
        'min_size_mb': 1000,
        'source_type': 'civitai',
        'model_version_id': '176425',
        'url': 'https://civitai.com/api/download/models/176425',
        'desc': 'SD1.5 chân dung Châu Á đẹp tự nhiên'
    },
    {
        'id': 'pony_diffusion_v6',
        'group': 'CHECKPOINTS',
        'is_fast': False,
        'name': 'Pony Diffusion V6 XL [SDXL Pony]',
        'category': 'checkpoints',
        'filename': 'ponyDiffusionV6XL_v6StartWithThisOne.safetensors',
        'aliases': ['ponydiffusion', 'pony_v6', 'ponydiffusionv6xl'],
        'min_size_mb': 2000,
        'source_type': 'civitai',
        'model_version_id': '290640',
        'url': 'https://civitai.com/api/download/models/290640',
        'desc': 'SDXL Pony V6 đa dạng phong cách & art'
    },
    # --- LORAS ---
    {
        'id': 'lora_krea_2_nsfw',
        'group': 'LORAS',
        'is_fast': True,
        'name': 'LoRA Krea 2 NSFW V4',
        'category': 'loras',
        'filename': 'krea_2_nsfw_v4.safetensors',
        'aliases': ['knp_000003000', 'krea_2_nsfw', 'knpv4'],
        'min_size_mb': 10,
        'source_type': 'civitai',
        'model_version_id': '3147117',
        'url': 'https://civitai.com/api/download/models/3147117',
        'desc': 'LoRA tăng chi tiết cho Krea 2 Turbo'
    },
    {
        'id': 'lora_expressive_h',
        'group': 'LORAS',
        'is_fast': False,
        'name': 'LoRA ExpressiveH (Hentai LoRa Style)',
        'category': 'loras',
        'filename': 'Expressive_H-000001.safetensors',
        'aliases': ['expressive_h', 'expressiveh'],
        'min_size_mb': 10,
        'source_type': 'civitai',
        'model_version_id': '382152',
        'url': 'https://civitai.com/api/download/models/382152',
        'desc': 'LoRA phong cách anime cảm xúc cao'
    },
    {
        'id': 'lora_breasts_out',
        'group': 'LORAS',
        'is_fast': False,
        'name': 'LoRA Breasts Out Fashion Clothing',
        'category': 'loras',
        'filename': '2BoutOval_K2v3_400.safetensors',
        'aliases': ['2boutoval', 'breasts_out', 'breastsout'],
        'min_size_mb': 5,
        'source_type': 'civitai',
        'model_version_id': '3134691',
        'url': 'https://civitai.com/api/download/models/3134691',
        'desc': 'LoRA phong cách thời trang gợi cảm'
    },
    # --- UTILITIES ---
    {
        'id': 'vae_sdxl',
        'group': 'UTILITIES',
        'is_fast': True,
        'name': 'SDXL VAE (Fix Màu & Khử Mờ)',
        'category': 'vae',
        'filename': 'sdxl_vae.safetensors',
        'aliases': ['sdxl_vae', 'sdxlvae'],
        'min_size_mb': 50,
        'source_type': 'hf',
        'repo_id': 'stabilityai/sdxl-vae',
        'hf_filename': 'sdxl_vae.safetensors',
        'desc': 'VAE chuẩn SDXL cân bằng màu'
    },
    {
        'id': 'upscaler_4x_ultrasharp',
        'group': 'UTILITIES',
        'is_fast': True,
        'name': 'Upscaler 4x-UltraSharp',
        'category': 'upscale_models',
        'filename': '4x-UltraSharp.pth',
        'aliases': ['4x-ultrasharp', '4x_ultrasharp', 'ultrasharp'],
        'min_size_mb': 10,
        'source_type': 'hf',
        'repo_id': 'lokcx/4x-Ultrasharp',
        'hf_filename': '4x-UltraSharp.pth',
        'desc': 'Nâng nét ảnh lên độ phân giải 4K'
    }
]

def find_existing_model(target_base, category, primary_name, aliases=None, min_mb=10):
    folder = os.path.join(target_base, category)
    if not os.path.exists(folder):
        return None, 0
    
    candidates = [primary_name.lower()]
    if aliases:
        for a in aliases:
            candidates.append(a.lower())
            
    min_bytes = min_mb * 1024 * 1024
    
    try:
        for fname in os.listdir(folder):
            full_path = os.path.join(folder, fname)
            if not os.path.isfile(full_path):
                continue
            if fname.endswith('.aria2'):
                continue
                
            sz = os.path.getsize(full_path)
            if sz < min_bytes:
                continue
                
            lower_fname = fname.lower()
            for cand in candidates:
                if cand == lower_fname or cand in lower_fname:
                    return full_path, sz
    except Exception:
        pass
        
    return None, 0

def link_model_file(src_file, dst_file):
    if not os.path.exists(src_file):
        return False
    try:
        os.makedirs(os.path.dirname(dst_file), exist_ok=True)
        if os.path.exists(dst_file) and not os.path.islink(dst_file) and os.path.samefile(src_file, dst_file):
            return True
        if not os.path.exists(dst_file):
            os.symlink(src_file, dst_file)
            return True
    except Exception:
        pass
    return False

# ==========================================================================
# CHẾ ĐỘ 1: KIỂM TRA TRẠNG THÁI TRÊN GOOGLE DRIVE (KHÔNG TẢI GÌ)
# ==========================================================================
if Che_Do.startswith('🔍'):
    print('=' * 88)
    print('📊 BẢNG KIỂM TRA MODEL TRÊN GOOGLE DRIVE')
    print(f'📂 Thư mục: {TARGET_BASE}')
    print('=' * 88)
    print(f"{'STT':<4} | {'TRẠNG THÁI':<12} | {'DANH MỤC':<15} | {'DUNG LƯỢNG':<10} | {'TÊN MODEL'}")
    print('-' * 88)
    
    count_exist = 0
    total_exist_bytes = 0
    current_grp = ''
    
    for idx, item in enumerate(CURATED_MODELS, 1):
        grp = item['group']
        if grp != current_grp:
            print(f'\n--- 🔹 NHÓM: {grp} ---')
            current_grp = grp
            
        existing_file, fsize = find_existing_model(TARGET_BASE, item['category'], item['filename'], item.get('aliases'), item['min_size_mb'])
        sub_folder = WEBUI_SUBDIR_MAP.get(item['category'], item['category'])
        target_dest_file = os.path.normpath(os.path.join(A1111_DIR, sub_folder, item['filename']))
        
        if existing_file:
            count_exist += 1
            total_exist_bytes += fsize
            sz_str = f"{fsize / (1024**3):.2f} GB" if fsize >= 1024**3 else f"{fsize / (1024**2):.1f} MB"
            status_str = '✅ ĐÃ CÓ'
            fname_display = os.path.basename(existing_file)
            link_model_file(existing_file, target_dest_file)
        else:
            sz_str = f"~{item['min_size_mb']} MB"
            status_str = '❌ CHƯA CÓ'
            fname_display = item['filename']
            
        tag_fast = ' [FAST CORE]' if item.get('is_fast') else ''
        print(f"{idx:<4} | {status_str:<12} | {item['category']:<15} | {sz_str:<10} | {item['name']}{tag_fast}")
        print(f"     ↳ File trên đĩa: {fname_display} ({item['desc']})")
    
    total_gb = total_exist_bytes / (1024**3)
    print('=' * 88)
    print(f'📈 TỔNG KẾT: Đã có {count_exist}/{len(CURATED_MODELS)} model trên Drive (Dung lượng: {total_gb:.2f} GB). Còn thiếu: {len(CURATED_MODELS) - count_exist} model.')
    print('💡 HƯỚNG DẪN:')
    print('   - Tải nhanh để chạy ngay: Chọn "⚡ Fast (Setup Cơ bản - Chạy nhanh nhất)" rồi bấm Play!')
    print('   - Tải tất cả model còn thiếu: Chọn "📦 Full (Tải trọn bộ tất cả pack còn thiếu)" rồi bấm Play!')
    print('=' * 88)

# ==========================================================================
# CHẾ ĐỘ 2: TIẾN TRÌNH TẢI MODEL (FAST HOẶC FULL - TỰ ĐỘNG BỎ QUA FILE ĐÃ CÓ)
# ==========================================================================
else:
    is_fast_mode = 'Fast' in Che_Do
    targets = [m for m in CURATED_MODELS if (m.get('is_fast') if is_fast_mode else True)]
    mode_name = '⚡ FAST MODE (Setup Cơ bản - Chỉ tải model cốt lõi để chạy ngay)' if is_fast_mode else '📦 FULL MODE (Tải tất cả các pack còn thiếu)'
    
    print('=' * 88)
    print(f'🚀 BẮT ĐẦU TIẾN TRÌNH TẢI: {mode_name}')
    print(f'🎯 Thư mục lưu trữ: {TARGET_BASE}')
    print('=' * 88)
    
    for item in targets:
        category = item['category']
        filename = item['filename']
        min_bytes = item['min_size_mb'] * 1024 * 1024
        dest_dir = os.path.join(TARGET_BASE, category)
        os.makedirs(dest_dir, exist_ok=True)
        dest_file = os.path.join(dest_dir, filename)
        sub_folder = WEBUI_SUBDIR_MAP.get(category, category)
        target_dest_file = os.path.normpath(os.path.join(A1111_DIR, sub_folder, filename))
        
        # 1. KIỂM TRA THÔNG MINH XEM ĐÃ CÓ FILE TRÊN DRIVE CHƯA
        existing_file, fsize = find_existing_model(TARGET_BASE, category, filename, item.get('aliases'), item['min_size_mb'])
        if existing_file:
            sz = fsize / (1024**3)
            print(f"⏩ [BỎ QUA - ĐÃ CÓ SẴN] {item['name']}")
            print(f"   Đã tìm thấy file: {os.path.basename(existing_file)} ({sz:.2f} GB) trên Drive -> BỎ QUA KHÔNG TẢI LẠI!")
            link_model_file(existing_file, target_dest_file)
            continue
            
        print(f"\n⬇️ [ĐANG TẢI MỚI] {item['name']}...")
        print(f"   Lưu vào: {dest_file}")
        
        # 2. Tải tùy theo nguồn (Civitai hoặc HuggingFace)
        if item['source_type'] == 'civitai':
            url = item['url']
            if Civitai_API_Token and 'token=' not in url:
                url += f'?token={Civitai_API_Token}' if '?' not in url else f'&token={Civitai_API_Token}'
                
            cmd = [
                'aria2c', '-c', '-x', '16', '-s', '16', '-k', '1M',
                '--file-allocation=none',
                '--header=User-Agent: Mozilla/5.0',
                '-d', dest_dir,
                '-o', filename,
                url
            ]
            ret = subprocess.run(cmd)
            if ret.returncode != 0 and 'alt_hf' in item:
                print('⚠️ Nguồn Civitai gặp sự cố, tự động chuyển sang HuggingFace dự phòng...')
                hf = item['alt_hf']
                try:
                    hf_hub_download(repo_id=hf['repo_id'], filename=hf['filename'], local_dir=dest_dir, local_dir_use_symlinks=False)
                    tmp = os.path.join(dest_dir, hf['filename'])
                    if tmp != dest_file and os.path.exists(tmp):
                        os.rename(tmp, dest_file)
                except Exception as err:
                    print(f'❌ HF dự phòng lỗi: {err}')
                    
        elif item['source_type'] == 'hf':
            try:
                dl_kwargs = {
                    'repo_id': item['repo_id'],
                    'filename': item['hf_filename'],
                    'local_dir': dest_dir,
                    'local_dir_use_symlinks': False
                }
                if HuggingFace_Token:
                    dl_kwargs['token'] = HuggingFace_Token
                dl_path = hf_hub_download(**dl_kwargs)
                if dl_path != dest_file and os.path.exists(dl_path):
                    if os.path.exists(dest_file):
                        os.remove(dest_file)
                    os.rename(dl_path, dest_file)
            except Exception as e:
                print(f"❌ Lỗi tải HuggingFace {filename}: {e}")
                
        # 3. Xác thực sau tải
        existing_file, fsize = find_existing_model(TARGET_BASE, category, filename, item.get('aliases'), item['min_size_mb'])
        if existing_file:
            sz = fsize / (1024**3)
            print(f"✅ Tải thành công: {filename} ({sz:.2f} GB)")
            link_model_file(existing_file, target_dest_file)
        else:
            print(f'⚠️ Cảnh báo: Tải {filename} chưa hoàn tất hoặc file bị lỗi.')
            
    print(f'\n🎉 Hoàn thành tiến trình cài đặt model ({mode_name})!')

# [![](https://img.shields.io/badge/Thiết%20kế-CookApps.net-0075ff)](https://cookapps.net) [![](https://img.shields.io/badge/Hướng%20dẫn-SD%20Nhập%20Môn-5858e6)](#) [![](https://img.shields.io/badge/Group-Support-0075ff)](#)

*`💡 Hỗ trợ CivitAI, Huggingface với link trực tiếp`*

In [ ]:
#@title 📥 4. Tải Thêm Model Tùy Chọn (CivitAI, Huggingface trực tiếp)
#@markdown Dán link model trực tiếp từ Civitai hoặc HuggingFace vào các ô tương ứng:
#@markdown - ⚡ **Bypass 403 Civitai**: Tự động nhận diện `modelVersionId` và nạp API Token từ Vault bảo mật.
#@markdown - 🛑 **Chống tải trùng thông minh**: Tự động kiểm tra trên Google Drive trước khi tải để bảo vệ dung lượng.
#@markdown - 🏷️ **Hỗ trợ đổi tên theo ý muốn**: Thêm `@=tên_muốn_đổi` vào cuối link (ví dụ: `https://civitai.com/models/...@=my_lora`).

Checkpoint__ = "" #@param {type:"string"}
Lora_Link___ = "" #@param {type:"string"}
Controlnet__ = "" #@param {type:"string"}
VAE_Link____ = "" #@param {type:"string"}

# Link tùy chọn khác (chọn thư mục lưu tương ứng)
Custom_Link_ = "" #@param {type:"string"}
Thu_Muc_Luu_ = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "embeddings"]

# --- MẬT KHẨU MÃ HÓA & API TOKENS ---
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}
Civitai_API_Token = "" #@param {type:"string"}
HuggingFace_Token = "" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

Civitai_API_Token = get_vault_token("civitai", Civitai_API_Token, Mat_Khau_Mo_Khoa)
HuggingFace_Token = get_vault_token("huggingface", HuggingFace_Token, Mat_Khau_Mo_Khoa)

import os
import re
import sys
import json
import urllib.request
import subprocess

!which aria2c > /dev/null || (apt-get update -qq && apt-get install -y -qq aria2)

DRIVE_DIR = "/content/drive/MyDrive/ComfyStudio/models"
A1111_DIR = "/content/stable-diffusion-webui/models"
has_drive = os.path.exists("/content/drive/MyDrive")
TARGET_BASE = DRIVE_DIR if has_drive else A1111_DIR

WEBUI_SUBDIR_MAP = {
    'checkpoints': 'Stable-diffusion',
    'loras': 'Lora',
    'vae': 'VAE',
    'controlnet': 'ControlNet',
    'upscale_models': 'ESRGAN',
    'embeddings': '../embeddings'
}

def split_string_at(link):
    delimiter = "@="
    if delimiter in link:
        parts = link.split(delimiter)
        name = parts[1].strip()
        if not any(name.endswith(ext) for ext in ['.safetensors', '.ckpt', '.pt', '.pth', '.bin']):
            name = f"{name}.safetensors"
        return parts[0].strip(), name
    return link.strip(), None

def resolve_model_info(url, token=""):
    headers = {"User-Agent": "Mozilla/5.0"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    
    if "huggingface.co" in url and "blob" in url:
        url = url.replace("/blob/", "/resolve/")
    
    if "civitai.com" in url or "civitai.red" in url:
        ver_match = re.search(r'modelVersionId=(\d+)', url)
        if ver_match:
            ver_id = ver_match.group(1)
            api_url = f"https://civitai.com/api/v1/model-versions/{ver_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    files = data.get("files", [])
                    fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                    size_kb = files[0].get("sizeKB", 0) if files else 0
                    dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                    if token:
                        dl_url += f"?token={token}"
                    return fname, dl_url, size_kb / 1024
            except Exception as e:
                print(f"⚠️ Không đọc được API Civitai: {e}")
                return f"{ver_id}.safetensors", f"https://civitai.com/api/download/models/{ver_id}?token={token}", 0
        
        model_match = re.search(r'/models/(\d+)', url)
        if model_match:
            model_id = model_match.group(1)
            api_url = f"https://civitai.com/api/v1/models/{model_id}"
            try:
                req = urllib.request.Request(api_url, headers=headers)
                with urllib.request.urlopen(req) as resp:
                    data = json.loads(resp.read().decode('utf-8'))
                    versions = data.get("modelVersions", [])
                    if versions:
                        ver = versions[0]
                        ver_id = ver.get("id")
                        files = ver.get("files", [])
                        fname = files[0].get("name") if files else f"{ver_id}.safetensors"
                        size_kb = files[0].get("sizeKB", 0) if files else 0
                        dl_url = f"https://civitai.com/api/download/models/{ver_id}"
                        if token:
                            dl_url += f"?token={token}"
                        return fname, dl_url, size_kb / 1024
            except Exception as e:
                print(f"⚠️ Không đọc được API Civitai: {e}")
                return f"{model_id}.safetensors", url, 0
                
    clean_url = url.split("?")[0]
    fname = os.path.basename(clean_url)
    return fname, url, 0

tasks = []
if Checkpoint__.strip():
    tasks.append(('checkpoints', Checkpoint__.strip()))
if Lora_Link___.strip():
    tasks.append(('loras', Lora_Link___.strip()))
if Controlnet__.strip():
    tasks.append(('controlnet', Controlnet__.strip()))
if VAE_Link____.strip():
    tasks.append(('vae', VAE_Link____.strip()))
if Custom_Link_.strip():
    tasks.append((Thu_Muc_Luu_, Custom_Link_.strip()))

if not tasks:
    print("ℹ️ Bạn chưa nhập link model nào vào các ô ở trên.")
    print("💡 Hãy dán link Checkpoint, LoRA, ControlNet hoặc Custom Link rồi bấm Play để tải đa luồng!")
else:
    print("=" * 80)
    print(f"🚀 BẮT ĐẦU TẢI {len(tasks)} MODEL TÙY CHỌN VÀO GOOGLE DRIVE")
    print(f"📂 Thư mục gốc: {TARGET_BASE}")
    print("=" * 80)
    
    for folder_cat, raw_link in tasks:
        clean_url, custom_name = split_string_at(raw_link)
        detected_name, download_url, estimated_mb = resolve_model_info(clean_url, Civitai_API_Token)
        final_filename = custom_name if custom_name else detected_name
        if not final_filename:
            final_filename = "downloaded_model.safetensors"
            
        target_dir = os.path.join(TARGET_BASE, folder_cat)
        os.makedirs(target_dir, exist_ok=True)
        dest_path = os.path.join(target_dir, final_filename)
        
        sub_folder = WEBUI_SUBDIR_MAP.get(folder_cat, folder_cat)
        target_dest_file = os.path.normpath(os.path.join(A1111_DIR, sub_folder, final_filename))
        
        # 1. Kiểm tra chống tải trùng
        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print(f"⏩ [BỎ QUA - ĐÃ CÓ SẴN] {final_filename} ({sz_str}) đã có trên Google Drive!")
            try:
                os.makedirs(os.path.dirname(target_dest_file), exist_ok=True)
                if not os.path.exists(target_dest_file):
                    os.symlink(dest_path, target_dest_file)
            except Exception:
                pass
            continue
            
        # 2. Tải đa luồng aria2c
        print(f"\n⬇️ [ĐANG TẢI] {final_filename}")
        print(f"   URL: {clean_url}")
        print(f"   Lưu vào: {dest_path}")
        
        cmd = [
            "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--file-allocation=none",
            "--header=User-Agent: Mozilla/5.0",
            "-d", target_dir,
            "-o", final_filename,
            download_url
        ]
        subprocess.run(cmd)
        
        # 3. Xác thực & symlink
        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 10 * 1024 * 1024:
            sz_bytes = os.path.getsize(dest_path)
            sz_str = f"{sz_bytes / (1024**3):.2f} GB" if sz_bytes >= 1024**3 else f"{sz_bytes / (1024**2):.1f} MB"
            print(f"🎉 TẢI THÀNH CÔNG: {final_filename} ({sz_str})")
            try:
                os.makedirs(os.path.dirname(target_dest_file), exist_ok=True)
                if not os.path.exists(target_dest_file):
                    os.symlink(dest_path, target_dest_file)
            except Exception:
                pass
        else:
            print(f"❌ Tải thất bại hoặc file không hợp lệ: {dest_path}")
            
    print("=" * 80)
    print("✅ Hoàn tất tiến trình tải model tùy chọn!")

In [ ]:
#@title 🚀 5. Khởi chạy SD WebUI Forge (Mở Link Truy cập)
#@markdown Chọn phương thức kết nối để lấy URL giao diện WebUI:

tunnel_type = "Gradio Share (Ch\u00ednh ch\u1ee7, d\u1ec5 d\u00f9ng nh\u1ea5t)" #@param ["Gradio Share (Ch\u00ednh ch\u1ee7, d\u1ec5 d\u00f9ng nh\u1ea5t)", "Ngrok Tunnel (Nhanh & \u1ed5n \u0111\u1ecbnh, c\u1ea7n Token)", "LocalTunnel (Mi\u1ec5n ph\u00ed)"]
ngrok_auth_token = "" #@param {type:"string"}
Mat_Khau_Mo_Khoa = "hf_ysYfQCoir" #@param {type:"string"}

# --- BẢO MẬT & MÃ HÓA TOKEN ---
ENCRYPTED_VAULT = "LqULmMTg4pq5Lmnzu8iaJXuKsm1qN6xaNrBtAUCzR7H9UzgBAaUwBn7bdhmVGz0iivY3JqGAoQb3sE4QKdWAvCh84P8odenY6urmVMgScgKzHZVm3vG6bl+GLZtdsfLrzN/87c4ioDk68aY3zHthib0hIo43jK9mFXbYEQbWl6K0wEaT80eWPyBY6bPULEc88beYeg3yQzTpr2NO3GdcM+li5yQi"

def unlock_vault(pwd: str) -> dict:
    if not pwd:
        return {}
    try:
        import hashlib, base64, json
        data = base64.b64decode(ENCRYPTED_VAULT.strip())
        key = pwd.encode('utf-8')
        keystream = bytearray()
        counter = 0
        while len(keystream) < len(data):
            keystream.extend(hashlib.sha256(key + counter.to_bytes(4, 'big')).digest())
            counter += 1
        dec = bytes([b ^ keystream[i] for i, b in enumerate(data)])
        res = json.loads(dec.decode('utf-8'))
        try:
            with open("/content/.vault_tokens.json", "w") as f:
                json.dump(res, f)
        except Exception:
            pass
        return res
    except Exception:
        return {}

def get_vault_token(name: str, manual: str = "", pwd: str = "") -> str:
    if manual and manual.strip():
        return manual.strip()
    try:
        import json
        if os.path.exists("/content/.vault_tokens.json"):
            cached = json.load(open("/content/.vault_tokens.json"))
            if name in cached and cached[name]:
                return cached[name]
    except Exception:
        pass
    if pwd:
        res = unlock_vault(pwd)
        if name in res and res[name]:
            return res[name]
    return ""

ngrok_auth_token = get_vault_token("ngrok", ngrok_auth_token, Mat_Khau_Mo_Khoa)
if ngrok_auth_token:
    print("🔓 [Bảo Mật] Đã tự động điền Ngrok Auth Token từ Mật khẩu giải mã!")

theme = "dark" #@param ["dark", "light"]

%cd /content/stable-diffusion-webui

launch_args = [
    "--theme", theme,
    "--enable-insecure-extension-access",
    "--cuda-malloc",
    "--cuda-stream",
    "--pin-shared-memory",
    "--no-half-vae",
    "--gradio-queue",
    "--outdir-txt2img-samples", "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111/txt2img",
    "--outdir-img2img-samples", "/content/drive/MyDrive/ComfyStudio/outputs/automatic1111/img2img"
]

if "gradio" in tunnel_type.lower():
    launch_args.append("--share")
elif "ngrok" in tunnel_type.lower():
    if not ngrok_auth_token:
        print("⚠️ Bạn chưa nhập ngrok_auth_token! Đang tự động chuyển sang --share...")
        launch_args.append("--share")
    else:
        launch_args.extend(["--ngrok", ngrok_auth_token])
elif "localtunnel" in tunnel_type.lower():
    import urllib.request
    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        print(f"🔑 Password / Endpoint IP cho LocalTunnel là: {ip}")
    except Exception:
        pass
    print("🌐 Đang kích hoạt LocalTunnel ở cổng 7860...")
    !npm install -g localtunnel > /dev/null 2>&1
    import subprocess
    subprocess.Popen(["lt", "--port", "7860"])

arg_string = " ".join(launch_args)
print(f"🚀 Đang khởi động WebUI với cờ: {arg_string}")
print("💡 Tìm dòng 'Running on public URL: https://...gradio.live' để nhấn vào mở giao diện WebUI!")

!python launch.py {arg_string}